<a href="https://colab.research.google.com/github/yomiannixs-jpg/AI-Optimization-app-using-LSTM_TensorFlow/blob/main/Iran_US_War_Oil_Stock_Predictability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Oil-Stock Predictability: Upload-Data Version
# Works in Google Colab or local Python
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ============================================================
# 1. Upload data in Google Colab
# ============================================================

try:
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]
except Exception:
    # For local use, put your file path here
    DATA_FILE = "oil_stock_data.csv"

OUTDIR = "oil_stock_outputs"
os.makedirs(OUTDIR, exist_ok=True)

WAR_START = "2026-03-01"
HORIZONS = [1, 5, 22]

# ============================================================
# 2. Load CSV or Excel
# ============================================================

if DATA_FILE.lower().endswith(".csv"):
    df = pd.read_csv(DATA_FILE)
elif DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE)
else:
    raise ValueError("Upload a CSV or Excel file.")

print("Loaded file:", DATA_FILE)
print("Columns found:", list(df.columns))

# Rename 'S&P500' column to 'SP500' if it exists
if "S&P500" in df.columns:
    df = df.rename(columns={"S&P500": "SP500"})

# ============================================================
# 3. REQUIRED COLUMN NAMES
# ============================================================
# Your file must contain these columns:
#
# Date
# SP500
# WTI
# Brent
#
# Optional for Table 6:
# VIX
# DXY
# UST10Y

required = ["Date", "SP500", "WTI", "Brent"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Rename your columns to exactly: {required}"
    )

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

for c in ["SP500", "WTI", "Brent", "VIX", "DXY", "UST10Y"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        # Ensure values are positive for columns that will be logged
        if c in ["SP500", "WTI", "Brent"]:
            df[c] = df[c].apply(lambda x: x if x > 0 else np.nan)

df = df.dropna(subset=["SP500", "WTI", "Brent"])

if df.empty:
    raise ValueError("Data are empty after cleaning. Check date/price columns.")

print("Usable observations:", len(df))
print(df.head())

# ============================================================
# 4. Construct variables
# ============================================================

df["WAR"] = (df.index >= pd.to_datetime(WAR_START)).astype(int)

df["SP500_ret"] = 100 * np.log(df["SP500"]).diff()
df["WTI_ret"] = 100 * np.log(df["WTI"]).diff()
df["Brent_ret"] = 100 * np.log(df["Brent"]).diff()

df["log_WTI"] = np.log(df["WTI"])
df["log_Brent"] = np.log(df["Brent"])

def pos_neg_cumsum(log_price):
    dlog = log_price.diff()
    pos = dlog.clip(lower=0).cumsum()
    neg = dlog.clip(upper=0).cumsum()
    return pos, neg

df["WTI_pos"], df["WTI_neg"] = pos_neg_cumsum(df["log_WTI"])
df["Brent_pos"], df["Brent_neg"] = pos_neg_cumsum(df["log_Brent"])

df["WTI_ret_pos"] = df["WTI_ret"].clip(lower=0)
df["WTI_ret_neg"] = df["WTI_ret"].clip(upper=0)
df["Brent_ret_pos"] = df["Brent_ret"].clip(lower=0)
df["Brent_ret_neg"] = df["Brent_ret"].clip(upper=0)

if "UST10Y" in df.columns:
    df["dUST10Y"] = df["UST10Y"].diff()

for h in HORIZONS:
    df[f"SR_fwd_{h}"] = 100 * (np.log(df["SP500"].shift(-h)) - np.log(df["SP500"]))

df = df.dropna(subset=[f"SR_fwd_{h}" for h in HORIZONS])

print("Final observations after creating forward returns:", len(df))

# ============================================================
# 5. Regression helper
# ============================================================

def nw_ols(y, X, lag):
    data = pd.concat([y, X], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if data.empty or len(data) < 30:
        raise ValueError(f"Regression data too small: {len(data)} observations.")

    y_clean = data.iloc[:, 0]
    X_clean = sm.add_constant(data.iloc[:, 1:], has_constant="add")

    res = sm.OLS(y_clean, X_clean).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": lag}
    )
    return res

def star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

def coef_fmt(res, var):
    if var not in res.params.index:
        return ""
    return f"{res.params[var]:.4f}{star(res.pvalues[var])}"

def t_fmt(res, var):
    if var not in res.tvalues.index:
        return ""
    return f"({res.tvalues[var]:.2f})"

# ============================================================
# 6. Figures
# ============================================================

def save_fig(series, title, ylabel, filename):
    plt.figure(figsize=(10, 4))
    plt.plot(series.index, series.values)
    plt.axvline(pd.to_datetime(WAR_START), linestyle="--")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, filename), dpi=300)
    plt.close()

save_fig(df["SP500"], "S&P 500 Index with War Start Date", "S&P 500", "fig_sp500.png")
save_fig(df["WTI"], "WTI Oil Price with War Start Date", "WTI Oil Price", "fig_wti.png")
save_fig(df["Brent"], "Brent Oil Price with War Start Date", "Brent Oil Price", "figA1_brent.png")
save_fig(df["SP500_ret"], "Daily S&P 500 Returns", "Daily return (%)", "figA2_sp500_returns.png")
save_fig(df["WTI_ret"], "Daily WTI Oil Returns", "Daily return (%)", "figA3_wti_returns.png")
save_fig(df["Brent_ret"], "Daily Brent Oil Returns", "Daily return (%)", "figA4_brent_returns.png")

# ============================================================
# 7. Export regression table helper
# ============================================================

def make_latex_reg_table(results, variables, caption, label, filename):
    lines = []
    lines.append(r"\begin{table}[!htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{threeparttable}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{8pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.20}")
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\toprule")
    lines.append(r"& \multicolumn{3}{c}{Dependent variable: Future S\&P 500 return} \\")
    lines.append(r"\cmidrule(lr){2-4}")
    lines.append(r"Variable & 1 day & 5 days & 22 days \\")
    lines.append(r"\midrule")

    for var, label_var in variables:
        coef_row = [label_var]
        t_row = [""]
        for h in HORIZONS:
            res = results[h]
            coef_row.append(coef_fmt(res, var))
            t_row.append(t_fmt(res, var))
        lines.append(" & ".join(coef_row) + r" \\")
        lines.append(" & ".join(t_row) + r" \[0.35em]")

    lines.append(r"\midrule")
    lines.append(
        "Observations & "
        + " & ".join(str(int(results[h].nobs)) for h in HORIZONS)
        + r" \\"
    )
    lines.append(
        "Adjusted $R^2$ & "
        + " & ".join(f"{results[h].rsquared_adj:.4f}" for h in HORIZONS)
        + r" \\"
    )
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\begin{tablenotes}[flushleft]")
    lines.append(r"\footnotesize")
    lines.append(
        r"\item \textit{Notes:} Newey--West $t$-statistics are reported in parentheses. "
        r"$^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$."
    )
    lines.append(r"\end{tablenotes}")
    lines.append(r"\end{threeparttable}")
    lines.append(r"\end{table}")

    path = os.path.join(OUTDIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved:", path)

# ============================================================
# 8. Table 1: baseline WTI
# ============================================================

results_t1 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OP": df["log_WTI"].shift(1),
        "WAR": df["WAR"],
        "OPxWAR": df["log_WTI"].shift(1) * df["WAR"]
    })
    results_t1[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t1,
    [
        ("OP", r"$OP_{t-1}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Baseline Wartime Predictive Regressions",
    "tab:baseline",
    "table1_baseline.tex"
)

# ============================================================
# 9. Table 2: nonlinear WTI
# ============================================================

results_t2 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OPpos": df["WTI_pos"].shift(1),
        "OPneg": df["WTI_neg"].shift(1),
        "WAR": df["WAR"],
        "OPposWAR": df["WTI_pos"].shift(1) * df["WAR"],
        "OPnegWAR": df["WTI_neg"].shift(1) * df["WAR"]
    })
    results_t2[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t2,
    [
        ("OPpos", r"$OP_{t-1}^{+}$Р"),
        ("OPneg", r"$OP_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPposWAR", r"$OP_{t-1}^{+}	imes WAR_t$Р"),
        ("OPnegWAR", r"$OP_{t-1}^{-}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Nonlinear Wartime Predictive Regressions",
    "tab:nonlinear",
    "table2_nonlinear.tex"
)

# ============================================================
# 10. Table 3: return-shock WTI
# ============================================================

results_t3 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "ORpos": df["WTI_ret_pos"].shift(1),
        "ORneg": df["WTI_ret_neg"].shift(1),
        "WAR": df["WAR"],
        "ORposWAR": df["WTI_ret_pos"].shift(1) * df["WAR"],
        "ORnegWAR": df["WTI_ret_neg"].shift(1) * df["WAR"]
    })
    results_t3[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t3,
    [
        ("ORpos", r"$OR_{t-1}^{+}$Р"),
        ("ORneg", r"$OR_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("ORposWAR", r"$OR_{t-1}^{+}	imes WAR_t$Р"),
        ("ORnegWAR", r"$OR_{t-1}^{-}	imes WAR_t$Р"),
        ("const", "Constant")
    ],
    "Positive and Negative Oil-Return Shock Robustness",
    "tab:return_shocks",
    "table3_return_shocks.tex"
)

# ============================================================
# 11. Table 6: macro controls, if columns exist
# ============================================================

macro_cols = ["VIX", "DXY", "dUST10Y"]

if all(c in df.columns for c in macro_cols):
    results_t6 = {}

    for h in HORIZONS:
        y = df[f"SR_fwd_{h}"]
        X = pd.DataFrame({
            "OP": df["log_WTI"].shift(1),
            "WAR": df["WAR"],
            "OPxWAR": df["log_WTI"].shift(1) * df["WAR"],
            "VIX": df["VIX"],
            "DXY": df["DXY"],
            "dUST10Y": df["dUST10Y"]
        })
        results_t6[h] = nw_ols(y, X, h)

    make_latex_reg_table(
        results_t6,
        [
            ("OP", r"$OP_{t-1}$Р"),
            ("WAR", r"$WAR_t$Р"),
            ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
            ("VIX", r"$VIX_t$Р"),
            ("DXY", r"$DXY_t$Р"),
            ("dUST10Y", r"$\Delta UST10Y_t$Р"),
            ("const", "Constant")
        ],
        "Incremental Predictive Information Beyond Macro-Financial Controls",
        "tab:incremental",
        "table6_incremental.tex"
    )
else:
    print("Skipping Table 6: missing one or more of VIX, DXY, UST10Y.")

# ============================================================
# 12. Save cleaned data
# ============================================================

df.to_csv(os.path.join(OUTDIR, "cleaned_oil_stock_data.csv"))

print("\nAll done.")
print("Outputs saved in:", OUTDIR)


Saving all_assets new.csv to all_assets new (1).csv
Loaded file: all_assets new (1).csv
Columns found: ['Date', 'WTI', 'Brent', 'S&P500', 'VIX', 'UST10Y', 'DXY']
Usable observations: 624
                  WTI      Brent       SP500        VIX       UST10Y    DXY
Date                                                                       
2024-01-02  75.889999  70.379997  102.199997  98.339890  4742.830078  13.20
2024-01-03  78.250000  72.699997  102.459999  98.116196  4704.810059  14.04
2024-01-04  77.589996  72.190002  102.419998  97.570427  4688.680176  14.13
2024-01-05  78.760002  73.809998  102.410004  97.221497  4697.240234  13.35
2024-01-08  76.120003  70.769997  102.209999  97.964111  4763.540039  13.08
Final observations after creating forward returns: 602
Saved: oil_stock_outputs/table1_baseline.tex
Saved: oil_stock_outputs/table2_nonlinear.tex
Saved: oil_stock_outputs/table3_return_shocks.tex
Saved: oil_stock_outputs/table6_incremental.tex

All done.
Outputs saved in: oil_stoc

In [ ]:
# ============================================================
# Oil-Stock Predictability: Upload-Data Version
# Works in Google Colab or local Python
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ============================================================
# 1. Upload data in Google Colab
# ============================================================

try:
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]
except Exception:
    # For local use, put your file path here
    DATA_FILE = "oil_stock_data.csv"

OUTDIR = "oil_stock_outputs"
os.makedirs(OUTDIR, exist_ok=True)

WAR_START = "2026-03-01"
HORIZONS = [1, 5, 22]

# ============================================================
# 2. Load CSV or Excel
# ============================================================

if DATA_FILE.lower().endswith(".csv"):
    df = pd.read_csv(DATA_FILE)
elif DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE)
else:
    raise ValueError("Upload a CSV or Excel file.")

print("Loaded file:", DATA_FILE)
print("Columns found:", list(df.columns))

# Rename 'S&P500' column to 'SP500' if it exists
if "S&P500" in df.columns:
    df = df.rename(columns={"S&P500": "SP500"})

# ============================================================
# 3. REQUIRED COLUMN NAMES
# ============================================================
# Your file must contain these columns:
#
# Date
# SP500
# WTI
# Brent
#
# Optional for Table 6:
# VIX
# DXY
# UST10Y

required = ["Date", "SP500", "WTI", "Brent"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Rename your columns to exactly: {required}"
    )

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

for c in ["SP500", "WTI", "Brent", "VIX", "DXY", "UST10Y"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        # Ensure values are positive for columns that will be logged
        if c in ["SP500", "WTI", "Brent"]:
            df[c] = df[c].apply(lambda x: x if x > 0 else np.nan)

df = df.dropna(subset=["SP500", "WTI", "Brent"])

if df.empty:
    raise ValueError("Data are empty after cleaning. Check date/price columns.")

print("Usable observations:", len(df))
print(df.head())

# ============================================================
# 4. Construct variables
# ============================================================

df["WAR"] = (df.index >= pd.to_datetime(WAR_START)).astype(int)

df["SP500_ret"] = 100 * np.log(df["SP500"]).diff()
df["WTI_ret"] = 100 * np.log(df["WTI"]).diff()
df["Brent_ret"] = 100 * np.log(df["Brent"]).diff()

df["log_WTI"] = np.log(df["WTI"])
df["log_Brent"] = np.log(df["Brent"])

def pos_neg_cumsum(log_price):
    dlog = log_price.diff()
    pos = dlog.clip(lower=0).cumsum()
    neg = dlog.clip(upper=0).cumsum()
    return pos, neg

df["WTI_pos"], df["WTI_neg"] = pos_neg_cumsum(df["log_WTI"])
df["Brent_pos"], df["Brent_neg"] = pos_neg_cumsum(df["log_Brent"])

df["WTI_ret_pos"] = df["WTI_ret"].clip(lower=0)
df["WTI_ret_neg"] = df["WTI_ret"].clip(upper=0)
df["Brent_ret_pos"] = df["Brent_ret"].clip(lower=0)
df["Brent_ret_neg"] = df["Brent_ret"].clip(upper=0)

if "UST10Y" in df.columns:
    df["dUST10Y"] = df["UST10Y"].diff()

for h in HORIZONS:
    df[f"SR_fwd_{h}"] = 100 * (np.log(df["SP500"].shift(-h)) - np.log(df["SP500"]))

df = df.dropna(subset=[f"SR_fwd_{h}" for h in HORIZONS])

print("Final observations after creating forward returns:", len(df))

# ============================================================
# 5. Regression helper
# ============================================================

def nw_ols(y, X, lag):
    data = pd.concat([y, X], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if data.empty or len(data) < 30:
        raise ValueError(f"Regression data too small: {len(data)} observations.")

    y_clean = data.iloc[:, 0]
    X_clean = sm.add_constant(data.iloc[:, 1:], has_constant="add")

    res = sm.OLS(y_clean, X_clean).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": lag}
    )
    return res

def star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

def coef_fmt(res, var):
    if var not in res.params.index:
        return ""
    return f"{res.params[var]:.4f}{star(res.pvalues[var])}"

def t_fmt(res, var):
    if var not in res.tvalues.index:
        return ""
    return f"({res.tvalues[var]:.2f})"

# ============================================================
# 6. Figures
# ============================================================

def save_fig(series, title, ylabel, filename):
    plt.figure(figsize=(10, 4))
    plt.plot(series.index, series.values)
    plt.axvline(pd.to_datetime(WAR_START), linestyle="--")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, filename), dpi=300)
    plt.close()

save_fig(df["SP500"], "S&P 500 Index with War Start Date", "S&P 500", "fig_sp500.png")
save_fig(df["WTI"], "WTI Oil Price with War Start Date", "WTI Oil Price", "fig_wti.png")
save_fig(df["Brent"], "Brent Oil Price with War Start Date", "Brent Oil Price", "figA1_brent.png")
save_fig(df["SP500_ret"], "Daily S&P 500 Returns", "Daily return (%)", "figA2_sp500_returns.png")
save_fig(df["WTI_ret"], "Daily WTI Oil Returns", "Daily return (%)", "figA3_wti_returns.png")
save_fig(df["Brent_ret"], "Daily Brent Oil Returns", "Daily return (%)", "figA4_brent_returns.png")

# ============================================================
# 7. Export regression table helper
# ============================================================

def make_latex_reg_table(results, variables, caption, label, filename):
    lines = []
    lines.append(r"\begin{table}[!htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{threeparttable}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{8pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.20}")
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\toprule")
    lines.append(r"& \multicolumn{3}{c}{Dependent variable: Future S\&P 500 return} \\")
    lines.append(r"\cmidrule(lr){2-4}")
    lines.append(r"Variable & 1 day & 5 days & 22 days \\")
    lines.append(r"\midrule")

    for var, label_var in variables:
        coef_row = [label_var]
        t_row = [""]
        for h in HORIZONS:
            res = results[h]
            coef_row.append(coef_fmt(res, var))
            t_row.append(t_fmt(res, var))
        lines.append(" & ".join(coef_row) + r" \\")
        lines.append(" & ".join(t_row) + r" \[0.35em]")

    lines.append(r"\midrule")
    lines.append(
        "Observations & "
        + " & ".join(str(int(results[h].nobs)) for h in HORIZONS)
        + r" \\"
    )
    lines.append(
        "Adjusted $R^2$ & "
        + " & ".join(f"{results[h].rsquared_adj:.4f}" for h in HORIZONS)
        + r" \\"
    )
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\begin{tablenotes}[flushleft]")
    lines.append(r"\footnotesize")
    lines.append(
        r"\item \textit{Notes:} Newey--West $t$-statistics are reported in parentheses. "
        r"$^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$."
    )
    lines.append(r"\end{tablenotes}")
    lines.append(r"\end{threeparttable}")
    lines.append(r"\end{table}")

    path = os.path.join(OUTDIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved:", path)

# ============================================================
# 8. Table 1: baseline WTI
# ============================================================

results_t1 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OP": df["log_WTI"].shift(1),
        "WAR": df["WAR"],
        "OPxWAR": df["log_WTI"].shift(1) * df["WAR"]
    })
    results_t1[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t1,
    [
        ("OP", r"$OP_{t-1}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Baseline Wartime Predictive Regressions",
    "tab:baseline",
    "table1_baseline.tex"
)

# ============================================================
# 9. Table 2: nonlinear WTI
# ============================================================

results_t2 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OPpos": df["WTI_pos"].shift(1),
        "OPneg": df["WTI_neg"].shift(1),
        "WAR": df["WAR"],
        "OPposWAR": df["WTI_pos"].shift(1) * df["WAR"],
        "OPnegWAR": df["WTI_neg"].shift(1) * df["WAR"]
    })
    results_t2[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t2,
    [
        ("OPpos", r"$OP_{t-1}^{+}$Р"),
        ("OPneg", r"$OP_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPposWAR", r"$OP_{t-1}^{+}	imes WAR_t$Р"),
        ("OPnegWAR", r"$OP_{t-1}^{-}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Nonlinear Wartime Predictive Regressions",
    "tab:nonlinear",
    "table2_nonlinear.tex"
)

# ============================================================
# 10. Table 3: return-shock WTI
# ============================================================

results_t3 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "ORpos": df["WTI_ret_pos"].shift(1),
        "ORneg": df["WTI_ret_neg"].shift(1),
        "WAR": df["WAR"],
        "ORposWAR": df["WTI_ret_pos"].shift(1) * df["WAR"],
        "ORnegWAR": df["WTI_ret_neg"].shift(1) * df["WAR"]
    })
    results_t3[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t3,
    [
        ("ORpos", r"$OR_{t-1}^{+}$Р"),
        ("ORneg", r"$OR_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("ORposWAR", r"$OR_{t-1}^{+}	imes WAR_t$Р"),
        ("ORnegWAR", r"$OR_{t-1}^{-}	imes WAR_t$Р"),
        ("const", "Constant")
    ],
    "Positive and Negative Oil-Return Shock Robustness",
    "tab:return_shocks",
    "table3_return_shocks.tex"
)

# ============================================================
# 11. Table 6: macro controls, if columns exist
# ============================================================

macro_cols = ["VIX", "DXY", "dUST10Y"]

if all(c in df.columns for c in macro_cols):
    results_t6 = {}

    for h in HORIZONS:
        y = df[f"SR_fwd_{h}"]
        X = pd.DataFrame({
            "OP": df["log_WTI"].shift(1),
            "WAR": df["WAR"],
            "OPxWAR": df["log_WTI"].shift(1) * df["WAR"],
            "VIX": df["VIX"],
            "DXY": df["DXY"],
            "dUST10Y": df["dUST10Y"]
        })
        results_t6[h] = nw_ols(y, X, h)

    make_latex_reg_table(
        results_t6,
        [
            ("OP", r"$OP_{t-1}$Р"),
            ("WAR", r"$WAR_t$Р"),
            ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
            ("VIX", r"$VIX_t$Р"),
            ("DXY", r"$DXY_t$Р"),
            ("dUST10Y", r"$\Delta UST10Y_t$Р"),
            ("const", "Constant")
        ],
        "Incremental Predictive Information Beyond Macro-Financial Controls",
        "tab:incremental",
        "table6_incremental.tex"
    )
else:
    print("Skipping Table 6: missing one or more of VIX, DXY, UST10Y.")

# ============================================================
# 12. Save cleaned data
# ============================================================

df.to_csv(os.path.join(OUTDIR, "cleaned_oil_stock_data.csv"))

print("\nAll done.")
print("Outputs saved in:", OUTDIR)


Saving all_assets new.csv to all_assets new (1).csv
Loaded file: all_assets new (1).csv
Columns found: ['Date', 'WTI', 'Brent', 'S&P500', 'VIX', 'UST10Y', 'DXY']
Usable observations: 624
                  WTI      Brent       SP500        VIX       UST10Y    DXY
Date                                                                       
2024-01-02  75.889999  70.379997  102.199997  98.339890  4742.830078  13.20
2024-01-03  78.250000  72.699997  102.459999  98.116196  4704.810059  14.04
2024-01-04  77.589996  72.190002  102.419998  97.570427  4688.680176  14.13
2024-01-05  78.760002  73.809998  102.410004  97.221497  4697.240234  13.35
2024-01-08  76.120003  70.769997  102.209999  97.964111  4763.540039  13.08
Final observations after creating forward returns: 602
Saved: oil_stock_outputs/table1_baseline.tex
Saved: oil_stock_outputs/table2_nonlinear.tex
Saved: oil_stock_outputs/table3_return_shocks.tex
Saved: oil_stock_outputs/table6_incremental.tex

All done.
Outputs saved in: oil_stoc

In [ ]:
# ============================================================
# Oil-Stock Predictability: Upload-Data Version
# Works in Google Colab or local Python
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ============================================================
# 1. Upload data in Google Colab
# ============================================================

try:
    from google.colab import files
    uploaded = files.upload()
    DATA_FILE = list(uploaded.keys())[0]
except Exception:
    # For local use, put your file path here
    DATA_FILE = "oil_stock_data.csv"

OUTDIR = "oil_stock_outputs"
os.makedirs(OUTDIR, exist_ok=True)

WAR_START = "2026-03-01"
HORIZONS = [1, 5, 22]

# ============================================================
# 2. Load CSV or Excel
# ============================================================

if DATA_FILE.lower().endswith(".csv"):
    df = pd.read_csv(DATA_FILE)
elif DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE)
else:
    raise ValueError("Upload a CSV or Excel file.")

print("Loaded file:", DATA_FILE)
print("Columns found:", list(df.columns))

# Rename 'S&P500' column to 'SP500' if it exists
if "S&P500" in df.columns:
    df = df.rename(columns={"S&P500": "SP500"})

# ============================================================
# 3. REQUIRED COLUMN NAMES
# ============================================================
# Your file must contain these columns:
#
# Date
# SP500
# WTI
# Brent
#
# Optional for Table 6:
# VIX
# DXY
# UST10Y

required = ["Date", "SP500", "WTI", "Brent"]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}\n"
        f"Rename your columns to exactly: {required}"
    )

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

for c in ["SP500", "WTI", "Brent", "VIX", "DXY", "UST10Y"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        # Ensure values are positive for columns that will be logged
        if c in ["SP500", "WTI", "Brent"]:
            df[c] = df[c].apply(lambda x: x if x > 0 else np.nan)

df = df.dropna(subset=["SP500", "WTI", "Brent"])

if df.empty:
    raise ValueError("Data are empty after cleaning. Check date/price columns.")

print("Usable observations:", len(df))
print(df.head())

# ============================================================
# 4. Construct variables
# ============================================================

df["WAR"] = (df.index >= pd.to_datetime(WAR_START)).astype(int)

df["SP500_ret"] = 100 * np.log(df["SP500"]).diff()
df["WTI_ret"] = 100 * np.log(df["WTI"]).diff()
df["Brent_ret"] = 100 * np.log(df["Brent"]).diff()

df["log_WTI"] = np.log(df["WTI"])
df["log_Brent"] = np.log(df["Brent"])

def pos_neg_cumsum(log_price):
    dlog = log_price.diff()
    pos = dlog.clip(lower=0).cumsum()
    neg = dlog.clip(upper=0).cumsum()
    return pos, neg

df["WTI_pos"], df["WTI_neg"] = pos_neg_cumsum(df["log_WTI"])
df["Brent_pos"], df["Brent_neg"] = pos_neg_cumsum(df["log_Brent"])

df["WTI_ret_pos"] = df["WTI_ret"].clip(lower=0)
df["WTI_ret_neg"] = df["WTI_ret"].clip(upper=0)
df["Brent_ret_pos"] = df["Brent_ret"].clip(lower=0)
df["Brent_ret_neg"] = df["Brent_ret"].clip(upper=0)

if "UST10Y" in df.columns:
    df["dUST10Y"] = df["UST10Y"].diff()

for h in HORIZONS:
    df[f"SR_fwd_{h}"] = 100 * (np.log(df["SP500"].shift(-h)) - np.log(df["SP500"]))

df = df.dropna(subset=[f"SR_fwd_{h}" for h in HORIZONS])

print("Final observations after creating forward returns:", len(df))

# ============================================================
# 5. Regression helper
# ============================================================

def nw_ols(y, X, lag):
    data = pd.concat([y, X], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if data.empty or len(data) < 30:
        raise ValueError(f"Regression data too small: {len(data)} observations.")

    y_clean = data.iloc[:, 0]
    X_clean = sm.add_constant(data.iloc[:, 1:], has_constant="add")

    res = sm.OLS(y_clean, X_clean).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": lag}
    )
    return res

def star(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

def coef_fmt(res, var):
    if var not in res.params.index:
        return ""
    return f"{res.params[var]:.4f}{star(res.pvalues[var])}"

def t_fmt(res, var):
    if var not in res.tvalues.index:
        return ""
    return f"({res.tvalues[var]:.2f})"

# ============================================================
# 6. Figures
# ============================================================

def save_fig(series, title, ylabel, filename):
    plt.figure(figsize=(10, 4))
    plt.plot(series.index, series.values)
    plt.axvline(pd.to_datetime(WAR_START), linestyle="--")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTDIR, filename), dpi=300)
    plt.close()

save_fig(df["SP500"], "S&P 500 Index with War Start Date", "S&P 500", "fig_sp500.png")
save_fig(df["WTI"], "WTI Oil Price with War Start Date", "WTI Oil Price", "fig_wti.png")
save_fig(df["Brent"], "Brent Oil Price with War Start Date", "Brent Oil Price", "figA1_brent.png")
save_fig(df["SP500_ret"], "Daily S&P 500 Returns", "Daily return (%)", "figA2_sp500_returns.png")
save_fig(df["WTI_ret"], "Daily WTI Oil Returns", "Daily return (%)", "figA3_wti_returns.png")
save_fig(df["Brent_ret"], "Daily Brent Oil Returns", "Daily return (%)", "figA4_brent_returns.png")

# ============================================================
# 7. Export regression table helper
# ============================================================

def make_latex_reg_table(results, variables, caption, label, filename):
    lines = []
    lines.append(r"\begin{table}[!htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\begin{threeparttable}")
    lines.append(r"\small")
    lines.append(r"\setlength{\tabcolsep}{8pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.20}")
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\toprule")
    lines.append(r"& \multicolumn{3}{c}{Dependent variable: Future S\&P 500 return} \\")
    lines.append(r"\cmidrule(lr){2-4}")
    lines.append(r"Variable & 1 day & 5 days & 22 days \\")
    lines.append(r"\midrule")

    for var, label_var in variables:
        coef_row = [label_var]
        t_row = [""]
        for h in HORIZONS:
            res = results[h]
            coef_row.append(coef_fmt(res, var))
            t_row.append(t_fmt(res, var))
        lines.append(" & ".join(coef_row) + r" \\")
        lines.append(" & ".join(t_row) + r" \[0.35em]")

    lines.append(r"\midrule")
    lines.append(
        "Observations & "
        + " & ".join(str(int(results[h].nobs)) for h in HORIZONS)
        + r" \\"
    )
    lines.append(
        "Adjusted $R^2$ & "
        + " & ".join(f"{results[h].rsquared_adj:.4f}" for h in HORIZONS)
        + r" \\"
    )
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\begin{tablenotes}[flushleft]")
    lines.append(r"\footnotesize")
    lines.append(
        r"\item \textit{Notes:} Newey--West $t$-statistics are reported in parentheses. "
        r"$^{***}p<0.01$, $^{**}p<0.05$, and $^{*}p<0.10$."
    )
    lines.append(r"\end{tablenotes}")
    lines.append(r"\end{threeparttable}")
    lines.append(r"\end{table}")

    path = os.path.join(OUTDIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved:", path)

# ============================================================
# 8. Table 1: baseline WTI
# ============================================================

results_t1 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OP": df["log_WTI"].shift(1),
        "WAR": df["WAR"],
        "OPxWAR": df["log_WTI"].shift(1) * df["WAR"]
    })
    results_t1[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t1,
    [
        ("OP", r"$OP_{t-1}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Baseline Wartime Predictive Regressions",
    "tab:baseline",
    "table1_baseline.tex"
)

# ============================================================
# 9. Table 2: nonlinear WTI
# ============================================================

results_t2 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OPpos": df["WTI_pos"].shift(1),
        "OPneg": df["WTI_neg"].shift(1),
        "WAR": df["WAR"],
        "OPposWAR": df["WTI_pos"].shift(1) * df["WAR"],
        "OPnegWAR": df["WTI_neg"].shift(1) * df["WAR"]
    })
    results_t2[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t2,
    [
        ("OPpos", r"$OP_{t-1}^{+}$Р"),
        ("OPneg", r"$OP_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("OPposWAR", r"$OP_{t-1}^{+}	imes WAR_t$Р"),
        ("OPnegWAR", r"$OP_{t-1}^{-}\times WAR_t$Р"),
        ("const", "Constant")
    ],
    "Nonlinear Wartime Predictive Regressions",
    "tab:nonlinear",
    "table2_nonlinear.tex"
)

# ============================================================
# 10. Table 3: return-shock WTI
# ============================================================

results_t3 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "ORpos": df["WTI_ret_pos"].shift(1),
        "ORneg": df["WTI_ret_neg"].shift(1),
        "WAR": df["WAR"],
        "ORposWAR": df["WTI_ret_pos"].shift(1) * df["WAR"],
        "ORnegWAR": df["WTI_ret_neg"].shift(1) * df["WAR"]
    })
    results_t3[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t3,
    [
        ("ORpos", r"$OR_{t-1}^{+}$Р"),
        ("ORneg", r"$OR_{t-1}^{-}$Р"),
        ("WAR", r"$WAR_t$Р"),
        ("ORposWAR", r"$OR_{t-1}^{+}	imes WAR_t$Р"),
        ("ORnegWAR", r"$OR_{t-1}^{-}	imes WAR_t$Р"),
        ("const", "Constant")
    ],
    "Positive and Negative Oil-Return Shock Robustness",
    "tab:return_shocks",
    "table3_return_shocks.tex"
)

# ============================================================
# 11. Table 6: macro controls, if columns exist
# ============================================================

macro_cols = ["VIX", "DXY", "dUST10Y"]

if all(c in df.columns for c in macro_cols):
    results_t6 = {}

    for h in HORIZONS:
        y = df[f"SR_fwd_{h}"]
        X = pd.DataFrame({
            "OP": df["log_WTI"].shift(1),
            "WAR": df["WAR"],
            "OPxWAR": df["log_WTI"].shift(1) * df["WAR"],
            "VIX": df["VIX"],
            "DXY": df["DXY"],
            "dUST10Y": df["dUST10Y"]
        })
        results_t6[h] = nw_ols(y, X, h)

    make_latex_reg_table(
        results_t6,
        [
            ("OP", r"$OP_{t-1}$Р"),
            ("WAR", r"$WAR_t$Р"),
            ("OPxWAR", r"$OP_{t-1}\times WAR_t$Р"),
            ("VIX", r"$VIX_t$Р"),
            ("DXY", r"$DXY_t$Р"),
            ("dUST10Y", r"$\Delta UST10Y_t$Р"),
            ("const", "Constant")
        ],
        "Incremental Predictive Information Beyond Macro-Financial Controls",
        "tab:incremental",
        "table6_incremental.tex"
    )
else:
    print("Skipping Table 6: missing one or more of VIX, DXY, UST10Y.")

# ============================================================
# 12. Save cleaned data
# ============================================================

df.to_csv(os.path.join(OUTDIR, "cleaned_oil_stock_data.csv"))

print("\nAll done.")
print("Outputs saved in:", OUTDIR)


Saving all_assets new.csv to all_assets new (1).csv
Loaded file: all_assets new (1).csv
Columns found: ['Date', 'WTI', 'Brent', 'S&P500', 'VIX', 'UST10Y', 'DXY']
Usable observations: 624
                  WTI      Brent       SP500        VIX       UST10Y    DXY
Date                                                                       
2024-01-02  75.889999  70.379997  102.199997  98.339890  4742.830078  13.20
2024-01-03  78.250000  72.699997  102.459999  98.116196  4704.810059  14.04
2024-01-04  77.589996  72.190002  102.419998  97.570427  4688.680176  14.13
2024-01-05  78.760002  73.809998  102.410004  97.221497  4697.240234  13.35
2024-01-08  76.120003  70.769997  102.209999  97.964111  4763.540039  13.08
Final observations after creating forward returns: 602
Saved: oil_stock_outputs/table1_baseline.tex
Saved: oil_stock_outputs/table2_nonlinear.tex
Saved: oil_stock_outputs/table3_return_shocks.tex
Saved: oil_stock_outputs/table6_incremental.tex

All done.
Outputs saved in: oil_stoc

In [ ]:
# ============================================================
# Table 6: Macro-financial controls
# ============================================================

if "UST10Y" in df.columns:
    df["dUST10Y"] = df["UST10Y"].diff()

macro_cols = ["VIX", "DXY", "dUST10Y"]

missing_macro = [c for c in macro_cols if c not in df.columns]

if missing_macro:
    raise ValueError(
        f"Cannot estimate Table 6. Missing macro columns: {missing_macro}. "
        "Your dataset must contain VIX, Dollar/DXY, and Treasury/UST10Y."
    )

results_t6 = {}

for h in HORIZONS:
    y = df[f"SR_fwd_{h}"]
    X = pd.DataFrame({
        "OP": df["log_WTI"].shift(1),
        "WAR": df["WAR"],
        "OPxWAR": df["log_WTI"].shift(1) * df["WAR"],
        "VIX": df["VIX"],
        "DXY": df["DXY"],
        "dUST10Y": df["dUST10Y"]
    })
    results_t6[h] = nw_ols(y, X, h)

make_latex_reg_table(
    results_t6,
    [
        ("OP", r"$OP_{t-1}$"),
        ("WAR", r"$WAR_t$"),
        ("OPxWAR", r"$OP_{t-1}\times WAR_t$"),
        ("VIX", r"$VIX_t$"),
        ("DXY", r"$DXY_t$"),
        ("dUST10Y", r"$\Delta UST10Y_t$"),
        ("const", "Constant")
    ],
    "Incremental Predictive Information Beyond Macro-Financial Controls",
    "tab:incremental",
    "table6_incremental.tex"
)

Saved: oil_stock_outputs/table6_incremental.tex


In [ ]:
import shutil
from google.colab import files

# Create a zip archive of the output directory
output_zip_path = shutil.make_archive(OUTDIR, 'zip', OUTDIR)

# Provide a download link for the zip file
files.download(output_zip_path)
print(f"All output files have been zipped and are available for download at {output_zip_path}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All output files have been zipped and are available for download at /content/oil_stock_outputs.zip
